# Runnable CNN Notebook for CIFAR-10

This notebook is designed to run cleanly in VS Code or Jupyter.

It includes:
- a clear hyperparameter section
- the CNN architecture inline
- the loss choice
- the training loop
- learning-curve plots
- a simple feature-map visualization

The defaults are set for a **quick run** so the notebook finishes in a reasonable time.
After that, you can increase the epochs and remove batch limits for a fuller training run.


## 1. Imports and Project Root

This cell makes sure the notebook can find the project files whether it is started from the project root or from the `Notebooks` folder.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import torch
from torch import nn

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'SRC').exists() and (candidate / 'Data').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root containing SRC/ and Data/.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from SRC.cifar10_data import CIFAR10_CLASSES, CIFAR10_MEAN, CIFAR10_STD, get_cifar10_dataloaders

print(f'Project root: {PROJECT_ROOT}')


## 2. Hyperparameters

This is the main place to edit the CNN experiment.

### Training hyperparameters
- `BATCH_SIZE`: how many images per update step
- `EPOCHS`: how many full passes through the training set
- `LEARNING_RATE`: how large each optimization step is
- `WEIGHT_DECAY`: regularization strength
- `USE_AUGMENTATION`: whether to apply simple image augmentation
- `MAX_TRAIN_BATCHES` and `MAX_VAL_BATCHES`: quick-run limits for notebook speed

### Architecture hyperparameters
- `CONV_CHANNELS`: number of feature maps in each CNN block
- `DROPOUT`: dropout before the final classifier


In [ ]:
# Quick notebook defaults
BATCH_SIZE = 128
EPOCHS = 1
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
USE_AUGMENTATION = False
NUM_WORKERS = 0
VALIDATION_SPLIT = 0.1
SEED = 42

# These limits keep the notebook quick. Set them to None for fuller training.
MAX_TRAIN_BATCHES = 20
MAX_VAL_BATCHES = 5

CONV_CHANNELS = (32, 64, 128)
DROPOUT = 0.4
NUM_CLASSES = 10


## 3. Load the Data

The dataloaders return training, validation, and test batches. The images are already normalized using the CIFAR-10 channel means and standard deviations.


In [ ]:
train_loader, val_loader, test_loader = get_cifar10_dataloaders(
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    augment=USE_AUGMENTATION,
    seed=SEED,
    num_workers=NUM_WORKERS,
)

images, labels = next(iter(train_loader))
print(f'Train batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')
print(f'Image batch shape: {tuple(images.shape)}')
print(f'Label batch shape: {tuple(labels.shape)}')
print(f'Classes: {CIFAR10_CLASSES}')


## 4. The Full CNN Model

This is the full first CNN baseline.

What it does:
- The first convolutional block learns low-level patterns like edges and color changes.
- The second block combines those into richer local patterns.
- The third block builds higher-level features.
- Pooling reduces spatial size.
- The final linear layer maps the learned representation to 10 class scores.


In [ ]:
class SimpleCIFAR10CNN(nn.Module):
    def __init__(self, conv_channels=(32, 64, 128), num_classes=10, dropout=0.4):
        super().__init__()
        c1, c2, c3 = conv_channels

        self.conv1_1 = nn.Conv2d(3, c1, kernel_size=3, padding=1)
        self.relu1_1 = nn.ReLU()
        self.conv1_2 = nn.Conv2d(c1, c1, kernel_size=3, padding=1)
        self.relu1_2 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2_1 = nn.Conv2d(c1, c2, kernel_size=3, padding=1)
        self.relu2_1 = nn.ReLU()
        self.conv2_2 = nn.Conv2d(c2, c2, kernel_size=3, padding=1)
        self.relu2_2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3_1 = nn.Conv2d(c2, c3, kernel_size=3, padding=1)
        self.relu3_1 = nn.ReLU()
        self.conv3_2 = nn.Conv2d(c3, c3, kernel_size=3, padding=1)
        self.relu3_2 = nn.ReLU()
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(c3, num_classes),
        )

    def forward(self, x):
        x = self.conv1_1(x)
        x = self.relu1_1(x)
        x = self.conv1_2(x)
        x = self.relu1_2(x)
        x = self.pool1(x)

        x = self.conv2_1(x)
        x = self.relu2_1(x)
        x = self.conv2_2(x)
        x = self.relu2_2(x)
        x = self.pool2(x)

        x = self.conv3_1(x)
        x = self.relu3_1(x)
        x = self.conv3_2(x)
        x = self.relu3_2(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x

    @torch.no_grad()
    def describe_feature_shapes(self, input_shape=(1, 3, 32, 32)):
        x = torch.zeros(input_shape)
        shapes = {'input': tuple(x.shape)}
        x = self.pool1(self.relu1_2(self.conv1_2(self.relu1_1(self.conv1_1(x)))))
        shapes['after_block1'] = tuple(x.shape)
        x = self.pool2(self.relu2_2(self.conv2_2(self.relu2_1(self.conv2_1(x)))))
        shapes['after_block2'] = tuple(x.shape)
        x = self.global_pool(self.relu3_2(self.conv3_2(self.relu3_1(self.conv3_1(x)))))
        shapes['after_block3'] = tuple(x.shape)
        x = self.classifier(x)
        shapes['logits'] = tuple(x.shape)
        return shapes


In [ ]:
model = SimpleCIFAR10CNN(conv_channels=CONV_CHANNELS, num_classes=NUM_CLASSES, dropout=DROPOUT)
print(model)
print(model.describe_feature_shapes())


## 5. Loss Choice

We use **cross-entropy loss** because CIFAR-10 is a multi-class classification problem with exactly one correct label per image.

Why it makes sense:
- the model outputs 10 logits
- the true target is one class out of 10
- cross-entropy rewards assigning high score to the correct class
- it penalizes confident wrong predictions strongly


In [ ]:
criterion = nn.CrossEntropyLoss()
criterion


## 6. Training and Evaluation Functions

These helper functions are kept inside the notebook so the whole workflow is self-contained.


In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, max_batches=None):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_examples = 0

    for batch_index, (inputs, targets) in enumerate(dataloader, start=1):
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_examples += batch_size

        if max_batches is not None and batch_index >= max_batches:
            break

    return {
        'loss': running_loss / running_examples,
        'accuracy': running_correct / running_examples,
    }


@torch.no_grad()
def evaluate(model, dataloader, criterion, device, max_batches=None):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_examples = 0

    for batch_index, (inputs, targets) in enumerate(dataloader, start=1):
        inputs = inputs.to(device)
        targets = targets.to(device)

        logits = model(inputs)
        loss = criterion(logits, targets)

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_examples += batch_size

        if max_batches is not None and batch_index >= max_batches:
            break

    return {
        'loss': running_loss / running_examples,
        'accuracy': running_correct / running_examples,
    }


## 7. Train the CNN

By default this notebook does a quick run so it stays responsive.

For a fuller run later, try:
- `EPOCHS = 5`
- `MAX_TRAIN_BATCHES = None`
- `MAX_VAL_BATCHES = None`


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)

model = SimpleCIFAR10CNN(conv_channels=CONV_CHANNELS, num_classes=NUM_CLASSES, dropout=DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = []
best_val_accuracy = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device, max_batches=MAX_TRAIN_BATCHES)
    val_metrics = evaluate(model, val_loader, criterion, device, max_batches=MAX_VAL_BATCHES)

    history.append({
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_accuracy': train_metrics['accuracy'],
        'val_loss': val_metrics['loss'],
        'val_accuracy': val_metrics['accuracy'],
    })

    if val_metrics['accuracy'] > best_val_accuracy:
        best_val_accuracy = val_metrics['accuracy']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_metrics['loss']:.4f} | train_acc={train_metrics['accuracy']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | val_acc={val_metrics['accuracy']:.4f}"
    )


## 8. Plot the Learning Curves

These plots help us see whether the model is learning and whether validation performance is improving alongside training performance.


In [ ]:
epoch_numbers = [entry['epoch'] for entry in history]
train_loss = [entry['train_loss'] for entry in history]
val_loss = [entry['val_loss'] for entry in history]
train_acc = [entry['train_accuracy'] for entry in history]
val_acc = [entry['val_accuracy'] for entry in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(epoch_numbers, train_loss, marker='o', label='Train loss')
axes[0].plot(epoch_numbers, val_loss, marker='o', label='Validation loss')
axes[0].set_title('CNN Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()

axes[1].plot(epoch_numbers, train_acc, marker='o', label='Train accuracy')
axes[1].plot(epoch_numbers, val_acc, marker='o', label='Validation accuracy')
axes[1].set_title('CNN Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.0, 1.0)
axes[1].legend()

plt.tight_layout()
plt.show()


## 9. Evaluate on the Test Set

We evaluate the best validation checkpoint on the held-out test set.


In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

test_metrics = evaluate(model, test_loader, criterion, device)
print(f'Best validation accuracy: {best_val_accuracy:.4f}')
print(f"Test loss: {test_metrics['loss']:.4f}")
print(f"Test accuracy: {test_metrics['accuracy']:.4f}")


## 10. Save a Summary

This stores the notebook results under `Data/cnn_notebook_runnable_runs/`.


In [ ]:
run_dir = PROJECT_ROOT / 'Data' / 'cnn_notebook_runnable_runs' / Path('run_placeholder')
run_dir = run_dir.parent / f"run_{torch.randint(100000, 999999, (1,)).item()}"
run_dir.mkdir(parents=True, exist_ok=True)

summary = {
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'dropout': DROPOUT,
    'conv_channels': CONV_CHANNELS,
    'augment': USE_AUGMENTATION,
    'max_train_batches': MAX_TRAIN_BATCHES,
    'max_val_batches': MAX_VAL_BATCHES,
    'best_val_accuracy': best_val_accuracy,
    'test_loss': test_metrics['loss'],
    'test_accuracy': test_metrics['accuracy'],
    'history': history,
}

summary_path = run_dir / 'summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'Saved summary to: {summary_path}')


## 11. Simple Feature-Map View

This looks at a few feature maps from the first convolutional layer.

These first-layer filters are the ones most likely to become edge or color-contrast detectors after training.


In [ ]:
def denormalize(image_tensor):
    mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR10_STD).view(3, 1, 1)
    return (image_tensor.cpu() * std + mean).clamp(0.0, 1.0)

sample_image, sample_label = test_loader.dataset[0]
sample_batch = sample_image.unsqueeze(0).to(device)

with torch.no_grad():
    conv1_maps = model.conv1_1(sample_batch).detach().cpu()
    logits = model(sample_batch).detach().cpu()

predicted_label = int(logits.argmax(dim=1).item())

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for index, axis in enumerate(axes.flat):
    axis.imshow(conv1_maps[0, index], cmap='viridis')
    axis.set_title(f'conv1_1 map {index}')
    axis.axis('off')

plt.tight_layout()
plt.show()

plt.figure(figsize=(4, 4))
plt.imshow(denormalize(sample_image).permute(1, 2, 0))
plt.title(f"True: {CIFAR10_CLASSES[sample_label]} | Predicted: {CIFAR10_CLASSES[predicted_label]}")
plt.axis('off')
plt.show()


## 12. How to Turn This Into a Full Run

After the notebook works end to end, switch to a fuller experiment by changing:

```python
EPOCHS = 5
MAX_TRAIN_BATCHES = None
MAX_VAL_BATCHES = None
USE_AUGMENTATION = True  # optional next improvement
```

That gives you a proper CNN experiment while keeping the same notebook structure.
